# Port-to-City candidate-pool audit

This notebook checks the committed run summary and bounded 100-record pool. It distinguishes direct place signals from wider waterfront, industrial, construction, and transit context. It does not promote model output into historical fact.

In [1]:
import json
from pathlib import Path

data_dir = Path('../data')
summary = json.loads((data_dir / 'run-summary-v1.json').read_text())
print(f"canonical rows: {summary['grains']['canonical_scored']['rows']:,}")
print(f"candidate rows: {summary['candidate_rows']}")
print(f"rows with place signals: {summary['place_signal_rows']}")
print(f"visual-family matches: {summary['grains']['visual_family']['matched']:,}")

canonical rows: 13,499
candidate rows: 100
rows with place signals: 123
visual-family matches: 841


In [2]:
rows = [json.loads(line) for line in (data_dir / 'candidate-pool-v1.jsonl').read_text().splitlines()]
exact_only = sum(bool(r['place_signals']['exact_source_supported']) and not r['place_signals']['model_inferred'] for r in rows)
both = sum(bool(r['place_signals']['exact_source_supported']) and bool(r['place_signals']['model_inferred']) for r in rows)
model_only = sum(not r['place_signals']['exact_source_supported'] and bool(r['place_signals']['model_inferred']) for r in rows)
fallback = len(rows) - exact_only - both - model_only
old_port = sum('old_port' in r['place_signals']['places'] for r in rows)
old_montreal = sum('old_montreal' in r['place_signals']['places'] for r in rows)
print(f'exact source only: {exact_only}')
print(f'exact source plus model: {both}')
print(f'model only: {model_only}')
print(f'contextual theme fallback: {fallback}')
print(f'old_port signal: {old_port}')
print(f'old_montreal signal: {old_montreal}')

exact source only: 52
exact source plus model: 12
model only: 13
contextual theme fallback: 23
old_port signal: 50
old_montreal signal: 32


## Interpretation

The bounded pool is deliberately wider than direct place-term retrieval: 23 rows enter through waterfront, industrial, construction, or transit themes. Those rows are discovery context, not place proof. The reviewed ten-record application core uses only E1 scene metadata or E2 exact report-sequence evidence for its main place claims.